In [1]:
# Import dependencies
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [2]:
# Load dataset
application_df = pd.read_csv("https://static.bc-edx.com/data/dl-1-2/m21/lms/starter/charity_data.csv")
application_df.head()


,EIN,NAME,APPLICATION_TYPE,AFFILIATION,CLASSIFICATION,USE_CASE,ORGANIZATION,STATUS,INCOME_AMT,SPECIAL_CONSIDERATIONS,ASK_AMT,IS_SUCCESSFUL
0,10520599,BLUE KNIGHTS MOTORCYCLE CLUB,T10,Independent,C1000,ProductDev,Association,1,0,N,5000,1
1,10531628,AMERICAN CHESAPEAKE CLUB CHARITABLE TR,T3,Independent,C2000,Preservation,Co-operative,1,1-9999,N,108590,1
2,10547893,ST CLOUD PROFESSIONAL FIREFIGHTERS,T5,CompanySponsored,C3000,ProductDev,Association,1,0,N,5000,0
3,10553066,SOUTHSIDE ATHLETIC ASSOCIATION,T3,CompanySponsored,C2000,Preservation,Trust,1,10000-24999,N,6692,1
4,10556103,GENETIC RESEARCH INSTITUTE OF THE DESERT,T3,Independent,C1000,Heathcare,Trust,1,100000-499999,N,142590,1


In [3]:
application_df = application_df.drop(columns=["EIN", "NAME"])


In [4]:
application_df.nunique()


APPLICATION_TYPE            17
AFFILIATION                  6
CLASSIFICATION              71
USE_CASE                     5
ORGANIZATION                 4
STATUS                       2
INCOME_AMT                   9
SPECIAL_CONSIDERATIONS       2
ASK_AMT                   8747
IS_SUCCESSFUL                2
dtype: int64

In [5]:
app_counts = application_df["APPLICATION_TYPE"].value_counts()
application_types_to_replace = app_counts[app_counts < 1000].index.tolist()

for app in application_types_to_replace:
    application_df["APPLICATION_TYPE"] = application_df["APPLICATION_TYPE"].replace(app, "Other")


In [6]:
class_counts = application_df["CLASSIFICATION"].value_counts()
classifications_to_replace = class_counts[class_counts < 1000].index.tolist()

for cls in classifications_to_replace:
    application_df["CLASSIFICATION"] = application_df["CLASSIFICATION"].replace(cls, "Other")


In [7]:
application_df = pd.get_dummies(application_df)


In [8]:
X = application_df.drop("IS_SUCCESSFUL", axis=1).values
y = application_df["IS_SUCCESSFUL"].values


In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)


In [10]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [11]:
# Define optimized model
nn_opt = tf.keras.models.Sequential()

# Input layer
nn_opt.add(tf.keras.layers.Input(shape=(X_train_scaled.shape[1],)))

# 1st hidden layer (more neurons)
nn_opt.add(tf.keras.layers.Dense(units=128, activation="relu"))

# 2nd hidden layer (added)
nn_opt.add(tf.keras.layers.Dense(units=64, activation="relu"))

# 3rd hidden layer (added)
nn_opt.add(tf.keras.layers.Dense(units=32, activation="relu"))

# Output layer
nn_opt.add(tf.keras.layers.Dense(units=1, activation="sigmoid"))

# Compile
nn_opt.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])


In [12]:
fit_opt = nn_opt.fit(X_train_scaled, y_train, epochs=150)


Epoch 1/150
804/804 ━━━━━━━━━━━━━━━━━━━━ 1s 421us/step - accuracy: 0.7080 - loss: 0.5824
Epoch 2/150
804/804 ━━━━━━━━━━━━━━━━━━━━ 0s 387us/step - accuracy: 0.7256 - loss: 0.5616
Epoch 3/150
804/804 ━━━━━━━━━━━━━━━━━━━━ 0s 385us/step - accuracy: 0.7336 - loss: 0.5503
Epoch 4/150
804/804 ━━━━━━━━━━━━━━━━━━━━ 0s 384us/step - accuracy: 0.7322 - loss: 0.5504
Epoch 5/150
804/804 ━━━━━━━━━━━━━━━━━━━━ 0s 384us/step - accuracy: 0.7256 - loss: 0.5548
Epoch 6/150
804/804 ━━━━━━━━━━━━━━━━━━━━ 0s 387us/step - accuracy: 0.7377 - loss: 0.5467
Epoch 7/150
804/804 ━━━━━━━━━━━━━━━━━━━━ 0s 395us/step - accuracy: 0.7324 - loss: 0.5527
Epoch 8/150
804/804 ━━━━━━━━━━━━━━━━━━━━ 0s 399us/step - accuracy: 0.7311 - loss: 0.5502
Epoch 9/150
804/804 ━━━━━━━━━━━━━━━━━━━━ 0s 392us/step - accuracy: 0.7328 - loss: 0.5503
Epoch 10/150
804/804 ━━━━━━━━━━━━━━━━━━━━ 0s 388us/step - accuracy: 0.7315 - loss: 0.5535
Epoch 11/150
804/804 ━━━━━━━━━━━━━━━━━━━━ 0s 394us/step - accuracy: 0.7320 - loss: 0.5493
Epoch 12/150
804/80

In [13]:
model_loss, model_accuracy = nn_opt.evaluate(X_test_scaled, y_test, verbose=2)
print(f"Loss: {model_loss}, Accuracy: {model_accuracy}")


268/268 - 0s - 467us/step - accuracy: 0.7310 - loss: 0.5995
Loss: 0.5995404124259949, Accuracy: 0.7309620976448059


In [14]:
nn_opt.save("AlphabetSoupCharity_Optimization.h5")
